# Fake Friends Data


## Overview
This notebook will show you how to create and query a table or DataFrame that you uploaded to DBFS. DBFS is a Databricks File System that allows you to store data for querying inside of Databricks. This notebook assumes that you have a file already inside of DBFS that you would like to read from.

This notebook is written in Python so the default cell type is Python. However, you can use different languages by using the %LANGUAGE syntax. Python, Scala, SQL, and R are all supported.



In [0]:

sc = spark.sparkContext  



In [0]:
# this will set the log level to ERROR. This will hide the INFO or WARNING messages that are printed out by default. If you want to see them, set this to INFO or WARN.
sc.setLogLevel("ERROR")

In [0]:
spark

SparkSession - hive 
 
 
 SparkContext 

 Spark UI 

 
 Version 
 v3.3.2 
 Master 
 local[8] 
 AppName 
 Databricks Shell

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
# see here for more info on the schema: https://spark.apache.org/docs/latest/sql-programming-guide.html#inferring-the-schema-using-reflection
# and here https://sparkbyexamples.com/pyspark/pyspark-sql-types-datatype-with-examples/

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("friendcount", IntegerType(), True),
    ])

friends = spark.read.csv('/FileStore/tables/fakefriends.csv', header=False, schema=schema)

# display the first 5 rows of the dataframe
friends.show(5)

+---+--------+---+-----------+
| id|    name|age|friendcount|
+---+--------+---+-----------+
|  0|    Will| 33|        385|
|  1|Jean-Luc| 26|          2|
|  2|    Hugh| 55|        221|
|  3|  Deanna| 40|        465|
|  4|   Quark| 68|         21|
+---+--------+---+-----------+
only showing top 5 rows



In [0]:
friends.createOrReplaceTempView("fakefriends_csv")

If running the cell below causes an error, you need to install the pyspark-magic module. Open a terminal (and make sure you have the spark conda environment active) and run the following command:
```pip install sparksql-magic```

NOTE: Remember, to activate the spark environment, run the following command:
```conda activate spark```

**NOTE2: You will need to restart your jupyter kernel after installing the module!!!!***

In [0]:
!pip install sparksql-magic



*** WARNING: max output size exceeded, skipping output. ***

     |████████████████████████████████| 317.0 MB 9.7 kB/s 
     |████████████████████████████████| 200 kB 54.3 MB/s 
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488511 sha256=de261413295ea2c21326caf550c129b7e9f1397696da297591d24f1023005c3a
  Stored in directory: /root/.cache/pip/wheels/92/09/11/aa01d01a7f005fda8a66ad71d2be7f8aa341bddafb27eee3c7
Successfully built pyspark
You should consider upgrading via the '/local_disk0/.ephemeral_nfs/envs/pythonEnv-e642a315-4ea7-463d-8cf2-62b1f75e1159/bin/python -m pip install --upgrade pip' command.


In [0]:
%load_ext sparksql_magic

In [0]:
%%sparksql
select * from fakefriends_csv

only showing top 20 row(s)


id,name,age,friendcount
0,Will,33,385
1,Jean-Luc,26,2
2,Hugh,55,221
3,Deanna,40,465
4,Quark,68,21
5,Weyoun,59,318
6,Gowron,37,220
7,Will,54,307
8,Jadzia,38,380
9,Hugh,27,181


In [0]:
%%sparksql 
select age, round(avg(friendcount), 1) As AvgFriendCount from fakefriends_csv group by age

only showing top 20 row(s)


age,AvgFriendCount
31,267.3
65,298.2
53,222.9
34,245.5
28,209.1
26,242.1
27,228.1
44,282.2
22,206.4
47,233.2


In [0]:
# since our friends data is only stored in volatile memory, let's save the table into our spark-warehouse
friends.write.saveAsTable("fake_friends", mode='overwrite')


In [0]:
spark.stop()